In [ ]:
# 필요한 라이브러리를 불러오고 Colab 보안 비밀에서 API 키를 가져오는 기본 설정

import requests
import pandas as pd
import time
import math

from google.colab import userdata

url = "https://apis.data.go.kr/6260000/IdlShrnProyService01/getIdlShrnProy01"

# Colab 보안 비밀에서 API 키 불러오기
SERVICE_KEY = userdata.get("BUSAN_API_KEY")

# 수집된 데이터와 실패 지역을 저장
all_rows = []
failed = []

In [ ]:
# ---------------------------------------------------------
# 특정 구·군의 유휴 공유재산 데이터를 수집하는 함수
# ---------------------------------------------------------
#
# collect_gugun("중구")
#
# 라고 실행하면 중구 데이터를 가져오고,
#
# collect_gugun("해운대구")
#
# 라고 실행하면 해운대구 데이터를 가져옴.


def collect_gugun(gugun):

    print(f"\n===== {gugun} 수집 시작 =====")


    # -----------------------------------------------------
    # STEP 1. API에 전달할 조건 설정
    # -----------------------------------------------------
    #
    # pageNo     : 몇 번째 페이지를 가져올지
    # numOfRows  : 한 페이지에 몇 건을 가져올지
    # gugun      : 어떤 구·군을 가져올지
    # resultType : JSON 형태로 결과를 받겠다는 의미

    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": 1,
        "numOfRows": 100,
        "gugun": gugun,
        "resultType": "json"
    }


    try:

        # -------------------------------------------------
        # STEP 2. API 서버에 요청
        # -------------------------------------------------
        #
        # timeout=30
        # → 서버가 30초 동안 응답하지 않으면
        #   무한정 기다리지 않고 요청을 중단

        response = requests.get(
            url,
            params=params,
            timeout=30
        )


        # 서버가 정상적으로 응답했는지 확인
        print("상태코드:", response.status_code)


        # -------------------------------------------------
        # STEP 3. HTTP 상태코드 확인
        # -------------------------------------------------
        #
        # 200 = 정상
        #
        # 404, 500, 504 등의 코드가 나오면
        # 정상적인 API 응답이 아니므로 해당 구·군은 중단

        if response.status_code != 200:

            print(f"❌ {gugun} HTTP 오류")

            # 실패한 구·군을 기록
            failed.append(gugun)

            return


        # -------------------------------------------------
        # STEP 4. 서버 응답을 JSON → Python 데이터로 변환
        # -------------------------------------------------

        data = response.json()


        # -------------------------------------------------
        # STEP 5. API 자체의 정상 응답 여부 확인
        # -------------------------------------------------
        #
        # 우리가 아까 정상 호출했을 때:
        #
        # resultCode = "00"
        # resultMsg  = "NORMAL_CODE"
        #
        # 였음.

        header = data["response"]["header"]

        if header["resultCode"] != "00":

            print("❌ API 오류:", header)

            failed.append(gugun)

            return


        # -------------------------------------------------
        # STEP 6. 해당 구·군의 전체 데이터 개수 확인
        # -------------------------------------------------

        body = data["response"]["body"]

        total_count = int(body["totalCount"])

        print("총 데이터 수:", total_count)


        # 데이터가 아예 없다면 여기서 종료
        if total_count == 0:
            print("⚠️ 데이터가 없습니다.")
            return


        # -------------------------------------------------
        # STEP 7. 필요한 페이지 수 계산
        # -------------------------------------------------
        #
        # 한 페이지당 100개씩 요청하고 있음.
        #
        # 예:
        #
        # 데이터 80개
        # → 1페이지
        #
        # 데이터 150개
        # → 2페이지
        #
        # 해운대구 246개
        # → 3페이지
        #
        # math.ceil()은 소수점을 올림하는 함수

        total_pages = math.ceil(total_count / 100)

        print("필요한 페이지 수:", total_pages)


        # -------------------------------------------------
        # STEP 8. 모든 페이지를 순서대로 가져오기
        # -------------------------------------------------

        for page in range(1, total_pages + 1):

            # 현재 가져올 페이지 번호 변경
            params["pageNo"] = page


            # 해당 페이지를 API에 요청
            response = requests.get(
                url,
                params=params,
                timeout=30
            )


            # -------------------------------------------------
            # 해당 페이지의 HTTP 상태 확인
            # -------------------------------------------------

            if response.status_code != 200:

                print(f"❌ {page}페이지 HTTP 오류")

                # 어느 구의 몇 페이지가 실패했는지 기록
                failed.append(f"{gugun}-{page}페이지")

                continue


            # JSON 데이터로 변환
            page_data = response.json()


            # -------------------------------------------------
            # 실제 유휴 공유재산 목록 꺼내기
            # -------------------------------------------------
            #
            # JSON 구조:
            #
            # response
            #   └ body
            #       └ items
            #           └ item
            #
            # 이 item 안에 실제 재산 목록이 들어 있음.

            items = (
                page_data["response"]["body"]
                .get("items", {})
                .get("item", [])
            )


            # -------------------------------------------------
            # 데이터가 딱 1개일 경우 처리
            # -------------------------------------------------
            #
            # API에 따라 데이터가 여러 개면 list,
            # 하나면 dict로 반환하는 경우가 있음.
            #
            # 이후 처리를 통일하기 위해
            # 하나여도 리스트 형태로 바꿈.

            if isinstance(items, dict):
                items = [items]


            # -------------------------------------------------
            # STEP 9. 가져온 데이터를 all_rows에 저장
            # -------------------------------------------------
            #
            # 기존 데이터 뒤에 이번 페이지 데이터를 추가

            all_rows.extend(items)


            print(
                f"✅ {page}/{total_pages} 페이지 완료 "
                f"({len(items)}건)"
            )


            # -------------------------------------------------
            # STEP 10. 서버에 너무 빠르게 요청하지 않도록
            # 1초 쉬기
            # -------------------------------------------------

            time.sleep(1)


        print(f"🎉 {gugun} 수집 완료")


    # -----------------------------------------------------
    # 실행 중 예상하지 못한 오류가 발생했을 경우
    # -----------------------------------------------------
    #
    # 프로그램 전체가 멈추는 대신
    # 실패한 구·군을 기록함.

    except Exception as e:

        print(
            f"❌ {gugun} 실패:",
            type(e).__name__,
            e
        )

        failed.append(gugun)

In [ ]:
# 부산 16개 구·군 중 첫 번째 4개만 수집

for gugun in [
    "중구",
    "서구",
    "동구",
    "영도구"
]:
    collect_gugun(gugun)

In [ ]:
# 두 번째 4개 구·군 수집

for gugun in [
    "부산진구",
    "동래구",
    "남구",
    "북구"
]:
    collect_gugun(gugun)

In [ ]:
# 실패한 남구·북구만 다시 수집

collect_gugun("남구")
collect_gugun("북구")

In [ ]:
# 남구·북구 API의 실제 응답 내용을 확인해서
# JSONDecodeError가 왜 나는지 원인을 확인하는 코드

for gugun in ["남구", "북구"]:

    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": 1,
        "numOfRows": 10,
        "gugun": gugun,
        "resultType": "json"
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    print(f"\n===== {gugun} =====")
    print("상태코드:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("응답 길이:", len(response.text))
    print("응답 앞부분:")
    print(repr(response.text[:1000]))

In [ ]:
# 세 번째 4개 구·군 수집

for gugun in [
    "해운대구",
    "사하구",
    "금정구",
    "강서구"
]:
    collect_gugun(gugun)

In [ ]:
# 마지막 4개 구·군 수집

for gugun in [
    "연제구",
    "수영구",
    "사상구",
    "기장군"
]:
    collect_gugun(gugun)

In [ ]:
# all_rows에 모아놓은 데이터를
# pandas의 DataFrame(표) 형태로 변환

df = pd.DataFrame(all_rows)


# 전체 데이터가 몇 행인지 확인
print("전체 행 수:", len(df))


# 각 구·군별로 몇 건씩 수집되었는지 확인
print("\n구·군별 건수:")

print(
    df["gugun"]
    .value_counts()
    .sort_index()
)


# 수집 과정에서 실패한 곳 확인
print("\n실패 목록:")

print(failed)

In [ ]:
# 지금까지 수집한 14개 구·군의 원본 데이터를 CSV로 저장

df = pd.DataFrame(all_rows)

df.to_csv(
    "부산_유휴공유재산_API원본_202605.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", len(df), "건")

In [ ]:
# 수집된 지역 수와 API에서 NODATA가 나온 지역을 확인

print("수집된 구·군 수:", df["gugun"].nunique())
print("전체 데이터 수:", len(df))

print("\n구·군별 데이터 수")
print(df["gugun"].value_counts().sort_index())

print("\n추가 확인 필요 지역:")
print(["남구", "북구"])

In [ ]:
# Google Drive를 Colab에 연결

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Google Drive에 저장해둔 유휴 공유재산 원본 데이터를 다시 불러오기

import pandas as pd

file_path = "/content/drive/MyDrive/부산_BigData_공모전/01_raw_data/유휴공유재산/부산_유휴공유재산_API원본_202605.csv"

df = pd.read_csv(file_path)

print("데이터 불러오기 완료")
print("전체 행 수:", len(df))
print("구·군 수:", df["gugun"].nunique())

## 구·군 공개자료 교차검증

- 사상구(2026.05)
  - 구청 공유재산 현황 104건
  - API 유휴 공유재산 104건
  - 주소 104/104 일치
  - 실면적과 API `zoneArea` 104/104 일치

- 서구
  - 구청 공유재산 현황(2025.10) 589건
  - API 유휴 공유재산(2026.05) 433건
  - API 433건의 주소가 구청 공개자료에 모두 포함됨
  - 두 자료의 기준시점이 달라 156건 차이의 원인은 확정할 수 없음

### 판단
남구·북구의 일반 공유재산 현황을 유휴 공유재산으로 바로 간주하지 않는다.
API에서 확보된 14개 구·군 자료를 우선 분석하고, 남구·북구는 NODATA로 별도 관리한다.

In [ ]:
# API 원본 데이터의 크기, 결측치, 중복, 주요 범주와 기준일자를 확인

print("데이터 크기:", df.shape)

print("\n[컬럼]")
print(df.columns.tolist())

print("\n[결측치]")
print(df.isnull().sum())

print("\n[완전 중복 행]")
print(df.duplicated().sum())

print("\n[재산용도]")
print(df["proyUsgNm"].value_counts(dropna=False))

print("\n[지목]")
print(df["landLdcgNm"].value_counts(dropna=False))

print("\n[기준일자]")
print(df["referenceDate"].value_counts(dropna=False).sort_index())

In [ ]:
# 결측·중복·이상값이 어느 구에서 발생하는지 구별로 확인

check = df.groupby("gugun").agg(
    전체건수=("lctnAddr", "size"),
    용도결측=("proyUsgNm", lambda x: x.isna().sum()),
    기준일결측=("referenceDate", lambda x: x.isna().sum()),
    주소고유수=("lctnAddr", "nunique")
)

check["중복가능건수"] = check["전체건수"] - check["주소고유수"]

print(check.sort_values("전체건수", ascending=False))

print("\n[숫자처럼 들어간 재산용도 샘플]")
mask = df["proyUsgNm"].astype(str).str.fullmatch(r"\d+")
print(
    df.loc[mask,
           ["gugun", "proyUsgNm", "lctnAddr",
            "landLdcgNm", "zoneArea", "referenceDate"]]
      .head(20)
)

print("\n[2026-09 이후 기준일자 샘플]")
date = pd.to_datetime(df["referenceDate"], errors="coerce")
print(
    df.loc[date >= "2026-09-01",
           ["gugun", "proyUsgNm", "lctnAddr",
            "landLdcgNm", "zoneArea", "referenceDate"]]
      .head(20)
)

In [ ]:
# 중복이 많은 4개 구에서 같은 주소의 행이 어떻게 반복되는지 확인

for gu in ["부산진구", "기장군", "해운대구", "강서구"]:
    temp = df[df["gugun"] == gu]

    dup = temp[temp.duplicated(
        subset=["lctnAddr"], keep=False
    )].sort_values("lctnAddr")

    print(f"\n===== {gu} =====")
    print("전체:", len(temp))
    print("주소 고유:", temp["lctnAddr"].nunique())
    print("완전히 동일한 중복:", temp.duplicated().sum())

    display(
        dup[
            ["gugun", "insttCode", "proyUsgNm", "lctnAddr",
             "landLdcgNm", "zoneArea", "referenceDate"]
        ].head(10)
    )

In [ ]:
# 연제구·동래구 공식 공개자료와 API의 주소·면적·지목 일치 여부를 검증

import pandas as pd
import re

# ---------- 연제구 ----------
yeonje = pd.read_excel(
    "/content/부산광역시_연제구_공유재산현황_20260522.xlsx",
    header=1
)

api_yeonje = df[df["gugun"] == "연제구"].copy()

# 주소 형식 통일: API에는 '부산광역시 연제구'가 없는 경우가 있어 제거해서 비교
def clean_addr(x):
    x = str(x).strip()
    x = x.replace("부산광역시 연제구 ", "")
    x = x.replace("번지", "")
    return re.sub(r"\s+", " ", x).strip()

yeonje["주소_정리"] = yeonje["소재지"].apply(clean_addr)
api_yeonje["주소_정리"] = api_yeonje["lctnAddr"].apply(clean_addr)

common_y = set(yeonje["주소_정리"]) & set(api_yeonje["주소_정리"])

print("===== 연제구 =====")
print("공식자료:", len(yeonje))
print("API:", len(api_yeonje))
print("주소 일치:", len(common_y))

y_compare = yeonje.merge(api_yeonje, on="주소_정리", how="inner")

y_compare["면적_공식"] = pd.to_numeric(y_compare["면적"], errors="coerce")
y_compare["면적_API"] = pd.to_numeric(y_compare["zoneArea"], errors="coerce")

print("면적 일치:",
      ((y_compare["면적_공식"] - y_compare["면적_API"]).abs() < 0.001).sum())

print("지목 일치:",
      (y_compare["지목"].astype(str).str.strip()
       == y_compare["landLdcgNm"].astype(str).str.strip()).sum())


# ---------- 동래구 ----------
dongnae = pd.read_csv(
    "/content/부산광역시 동래구_공유재산 현황_20260701.csv",
    encoding="cp949"
)

api_dongnae = df[df["gugun"] == "동래구"].copy()

def clean_addr_dongnae(x):
    x = str(x).strip()
    x = x.replace("부산광역시 동래구 ", "")
    return re.sub(r"\s+", " ", x).strip()

dongnae["주소_정리"] = dongnae["소재지"].apply(clean_addr_dongnae)
api_dongnae["주소_정리"] = api_dongnae["lctnAddr"].apply(clean_addr_dongnae)

common_d = set(dongnae["주소_정리"]) & set(api_dongnae["주소_정리"])

print("\n===== 동래구 =====")
print("공식자료:", len(dongnae))
print("API:", len(api_dongnae))
print("주소 일치:", len(common_d))

d_compare = dongnae.merge(api_dongnae, on="주소_정리", how="inner")

d_compare["면적_공식"] = pd.to_numeric(
    d_compare["면적(제곱미터)"], errors="coerce"
)
d_compare["면적_API"] = pd.to_numeric(
    d_compare["zoneArea"], errors="coerce"
)

print("면적 일치:",
      ((d_compare["면적_공식"] - d_compare["면적_API"]).abs() < 0.001).sum())

In [ ]:
# 핵심 분석 변수의 형식과 이상치를 최종 점검

area = pd.to_numeric(df["zoneArea"], errors="coerce")

print("===== 면적 =====")
print(area.describe())
print("숫자 변환 실패:", area.isna().sum())
print("0 이하:", (area <= 0).sum())

print("\n[면적이 작은 재산]")
print(
    df.assign(zoneArea_num=area)
      .nsmallest(20, "zoneArea_num")
      [["gugun", "lctnAddr", "landLdcgNm", "zoneArea_num"]]
)

print("\n[면적이 큰 재산]")
print(
    df.assign(zoneArea_num=area)
      .nlargest(20, "zoneArea_num")
      [["gugun", "lctnAddr", "landLdcgNm", "zoneArea_num"]]
)

print("\n===== 지목 =====")
print(df["landLdcgNm"].value_counts())

print("\n===== 주소 =====")
print("주소 결측:", df["lctnAddr"].isna().sum())
print("주소 공백:", df["lctnAddr"].astype(str).str.strip().eq("").sum())

print("\n[구·군명이 주소에 없는 건수]")
for gu in sorted(df["gugun"].unique()):
    temp = df[df["gugun"] == gu]
    missing_gu = ~temp["lctnAddr"].astype(str).str.contains(gu, regex=False)
    print(gu, ":", missing_gu.sum(), "/", len(temp))

In [ ]:
# 검증이 끝난 원본에서 분석용 데이터셋을 생성
# 완전 동일 중복만 제거하고 주소·지목·면적 형식을 표준화

df_clean = df.drop_duplicates().copy().reset_index(drop=True)

# 면적 숫자형
df_clean["zoneArea"] = pd.to_numeric(
    df_clean["zoneArea"], errors="coerce"
)

# 연제구 주소 표준화
mask = (
    (df_clean["gugun"] == "연제구")
    & ~df_clean["lctnAddr"].str.contains("부산광역시 연제구", na=False)
)

df_clean.loc[mask, "lctnAddr"] = (
    "부산광역시 연제구 "
    + df_clean.loc[mask, "lctnAddr"].str.replace("번지", "", regex=False)
)

# 지목 코드 제거 및 명칭 통일
df_clean["land_type"] = (
    df_clean["landLdcgNm"]
    .astype(str)
    .str.replace(r"^\d{2}-", "", regex=True)
    .str.strip()
    .replace({
        "대지": "대"
    })
)

print("원본:", len(df))
print("완전 중복 제거:", len(df) - len(df_clean))
print("분석용:", len(df_clean))

print("\n[구·군별]")
print(df_clean["gugun"].value_counts())

print("\n[표준화 지목]")
print(df_clean["land_type"].value_counts())

print("\n[0㎡ 이하]")
print(
    df_clean.loc[
        df_clean["zoneArea"] <= 0,
        ["gugun", "lctnAddr", "land_type", "zoneArea"]
    ]
)

## 데이터 품질검증 및 1차 정제 결과

- API 원본: 14,594건
- 완전 동일 중복: 5,190건
- 중복 제거 후 분석 대상: 9,404건
- 주소, 지목, 면적 핵심 변수는 공식 구청 공개자료와 교차검증
- 사상구: 주소 104/104, 실면적 104/104 일치
- 연제구: 주소 151/151, 지목 151/151, 면적 150/151 일치
- 동래구: 주소 230/230, 면적 230/230 일치
- 서구: API 433건 모두 구청 공유재산 공개자료에 포함

### 정제
- 완전히 동일한 행만 중복 제거
- 지목 코드 표기 통일 (예: 08-대 → 대, 14-도로 → 도로)
- 연제구 주소 형식 표준화
- 원본 컬럼은 보존

### 주의사항
- 남구·북구는 API NODATA로 현재 분석 데이터에 미포함
- `proyUsgNm`, `referenceDate`는 구·군별 품질 차이로 핵심 분석변수에서 제외
- 사하구 0㎡ 1건은 삭제하지 않고 추후 활용 가능성 필터링 단계에서 검토
- 극소·대규모 면적은 데이터 오류로 간주하지 않고 실제 활용 적합성 분석에서 판단

In [ ]:
# 검증 및 기본 정제가 끝난 분석용 데이터를 Drive에 저장

save_path = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/유휴공유재산_1차정제.csv"

df_clean.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", save_path)

# 유휴 공유재산 데이터 수집·품질검증 결과

## 1. 데이터 수집

부산광역시 구·군 유휴 공유재산 현황 API를 이용하여 데이터를 수집하였다.

- API 원본: 14,594건
- API 제공 구·군: 14개
- 남구·북구: API 조회 시 NODATA(030) 응답
- 남구·북구는 실제 유휴재산이 0건이라고 판단하지 않고 API 미제공 지역으로 별도 관리한다.

## 2. 공식자료 교차검증

API 핵심 변수의 신뢰성을 확인하기 위해 구·군 공식 공유재산 공개자료와 비교하였다.

- 사상구
  - 공식자료 104건 / API 104건
  - 주소 104/104 일치
  - 실면적과 API 면적 104/104 일치

- 연제구
  - 공식자료 151건 / API 151건
  - 주소 151/151 일치
  - 지목 151/151 일치
  - 면적 150/151 일치

- 동래구
  - 공식자료 230건 / API 230건
  - 주소 230/230 일치
  - 면적 230/230 일치

- 서구
  - 공식자료(2025.10) 589건 / API(2026.05) 433건
  - API 433건의 주소가 공식자료에 모두 포함됨
  - 두 자료의 기준시점이 달라 건수 차이의 원인은 확정하지 않음

## 3. 데이터 품질검증

원본 14,594건 중 완전히 동일한 중복 5,190건을 확인하였다.

중복 제거 후 분석 대상은 9,404건이다.

핵심 분석변수인 다음 항목에서는 결측이 확인되지 않았다.

- 구·군 (`gugun`)
- 소재지주소 (`lctnAddr`)
- 표준화 지목 (`land_type`)
- 면적 (`zoneArea`)

`proyUsgNm`(재산용도명)과 `referenceDate`(기준일자)는
구·군별 결측 및 비정상적인 값이 확인되어 핵심 분석변수로 사용하지 않는다.

## 4. 1차 정제

- 완전히 동일한 중복행 제거
- 지목 코드 및 명칭 표준화
  - 예: `08-대 → 대`, `14-도로 → 도로`
- 연제구 주소 형식 표준화
- 원본 변수는 추적 가능하도록 보존

사하구에서 면적 0㎡인 재산 1건이 확인되었으나,
현재 단계에서는 삭제하지 않고 향후 활용 가능성 검토 단계에서 판단한다.

극소면적 또는 대규모 재산 역시 단순 이상치로 판단하여 제거하지 않는다.

## 5. 분석용 데이터

최종 1차 정제 데이터: 9,404건

`유휴공유재산_1차정제.csv`

이 데이터를 이후 유휴 공유재산 EDA 및 활용 후보 분석의 기본 데이터로 사용한다.